# Financial Analysis Demo with Cash

This notebook demonstrates the capabilities of the `cash` library for caching expensive data science operations. We will work with a large, deterministically generated financial dataset.

In [ ]:
import cash
import pandas as pd
import numpy as np
import time
import os
print(os.getcwd())
#os.chdir(r'C:\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples')

In [ ]:
%cash_on
#%cash_debug on

## 1. Data Loading
Loading a large CSV file can be slow. With `cash`, this operation is cached after the first run.

In [ ]:
print(os.getcwd())
# Ensure the data exists (it should have been generated by generate_financial_data.py)
data_path = 'large_financial_data.csv'
#if not os.path.exists(data_path):
    #print("Data file not found! Please run generate_financial_data.py first.")
#else:
print("Loading data...")
# This read_csv call will be cached
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
print(df.head())

## 2. Preprocessing
Basic sorting and cleaning.

In [ ]:
print("Sorting data...")
t0 = time.time()
df = df.sort_values(by=['Ticker', 'Date'])
print(f"Sorted in {time.time() - t0:.2f}s")

In [ ]:
df #print

## 3. Heavy Computation (Statement-wise Caching)
Here we perform multiple heavy calculations using custom rolling window functions. These are significantly slower than vectorized pandas operations, making them perfect candidates for caching. `cash` caches these statement-wise.

**Try this:** Run the cell once. Then change the window size in the first statement (SMA) and run it again. You'll see the second statement (Voladj) loads instantly from cache!

In [ ]:
print("Calculating Volatility Adjusted Mean (Statement 1)....")
t0 = time.time()
# Heavy operation 1: Another slow rolling application
df['VolAdj_20'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=20).apply(lambda y: np.mean(y) / (np.std(y) + 1e-6), raw=True))
print(f"VolAdj calculated in {time.time() - t0:.2f}s")

print("Calculating Weighted SMA (Statement 2)...")
t0 = time.time()
# Heavy operation 2: Custom weighted mean using apply() (slow)
def custom_weighted_mean(x):
    weights = np.arange(1, len(x) + 1)
    return np.sum(x * weights) / np.sum(weights)

df['SMA_71'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=50).apply(custom_weighted_mean, raw=True))
print(f"SMA calculated  in {time.time() - t0:.2f}s")


df

In [ ]:
#print(x)
df

In [ ]:
%cash_provenance df --graph   

## 4. More Metrics
Adding RSI calculation in a separate cell.

In [ ]:
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

print("Calculating RSI...")
t0 = time.time()
df['RSI'] = df.groupby('Ticker')['Close'].transform(calculate_rsi)
print(f"RSI calculated in {time.time() - t0:.2f}s")
df

## 5. Aggregation & Analysis

In [ ]:
summary = df.groupby('Ticker').agg({
    'Close': ['mean', 'std'],
    'Volume': 'sum',
    'RSI': 'mean',
    'SMA_50': 'last'
})
print(summary)

In [ ]:
print("hi")
print(df)

## 6. Loop Caching Demo

Cash can now cache **individual iterations** of loops! Each iteration is cached separately based on:
- The loop variable value
- Dependencies accessed in that iteration

This means if you change earlier iterations, later iterations that are independent can still be restored from cache.

In [ ]:
# Processing each ticker separately in a loop
# Each iteration is cached independently!
ticker_stats = {}
print(df)
a = []

for ticker in ["TSLA", "GOOGL", "AAPL", "AMZN"]:
    ticker_data = df[df["Ticker"] == ticker]
    stats = {
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
    }
    ticker_stats[ticker] = stats
    print(f"{ticker}: mean={stats['mean_close'][0]:.2f}, std={stats['std_close']:.2f}")
    a.append(ticker)

print(ticker_stats.keys())
print("\nDone processing all tickers!")


In [ ]:
ticker_data

In [ ]:
{
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
}

In [ ]:
ticker_stats.keys()
a

### Conditional Caching

Cash also caches **branches of if statements** - only the executed branch is stored, making cache keys more precise.

In [ ]:
# Conditional processing based on data size
row_count = len(df)

if row_count > 500000:
    # Heavy processing for large datasets
    sample = df.sample(n=10000, random_state=42)
    report_type = "sampled"
    print(f"Large dataset ({row_count:,} rows) - using sampled analysis")
else:
    # Full processing for smaller datasets
    sample = df.copy()
    report_type = "full"
    print(f"Small dataset ({row_count:,} rows) - using full analysis")

print(f"Report type: {report_type}, sample size: {len(sample):,}")

### Nested Loops Example

Even nested loops work - each combination of outer/inner loop variables gets its own cache entry.

In [ ]:
# Compute metrics for multiple tickers across different time windows
windows = [10, 20, 50]
tickers = ["AAPL", "GOOGL"]

results = {}
for ticker in tickers:
    results[ticker] = {}
    ticker_data = df[df["Ticker"] == ticker]["Close"]
    for window in windows:
        sma = ticker_data.rolling(window=window).mean().iloc[-1]
        results[ticker][f"SMA_{window}"] = sma
        print(f"{ticker} SMA-{window}: {sma:.2f}")

print("\nAll window calculations complete!")

In [ ]:
for a, b in zip([1, 2, 3, 4, 5, 6, 7], ['x', 'y', 'z', 'u', 'v', 'w', 't']):
    print(f"{a} - {b}")

In [ ]:
try:
    print("hi")
    asdf = 235
    raise ValueError("This is a test error")
    x = 123
except ValueError as e:
    print(f"Value error occurred: {e}")
    time.sleep(1)  # Simulate some cleanup time

In [ ]:
c = 122

In [ ]:

def f(a):
    return c + a

x = {'b': 123}
x['a'] = f(0)
print(x)

In [ ]:
x

In [ ]:
import sys
sys.path.append("examples")
import metrics
print("Testing metrics module...")
print(metrics.increment(5))

In [ ]:
class Test:
    def __init__(self):
        self.x = 1
        self.y = 2

a = Test()
print(a.x)

a.x = 126

print(a.x)

In [ ]:
a.x = 1290

In [ ]:
print(a.x)
a.x

In [ ]:
print("hi")
raise ValueError("This is a test error to demonstrate error handling in the notebook.")
print("ho")

In [ ]:
@cash.cache
def dep(a):
    import time
    time.sleep(1)
    return a + 1

In [ ]:
@cash.cache
def fun(a, b):
    import time
    time.sleep(1)
    return a + b + dep(a)

x = 6
[fun(x, i % 3) for i in range(52)]

In [ ]:
import metrics

print(metrics.super_fun(10))
sum([metrics.fun(x, i % 3) for i in range(51)])

In [ ]:
@cash.cache
def super_fun(df):
    import time
    time.sleep(1)  # Simulate a heavy operation
    return df.iloc[1]

In [ ]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6]
})
print(super_fun(df), "", "")